## ▶ What you'll see when you run this
- The **same query** ranked three ways — **dense** (embeddings), **BM25** (keyword), and a **fused hybrid** list — then a **cross-encoder reranker** reordering the top candidates, so you can watch each upgrade change the order.

**Time:** ~8 min · **Cost:** free (all local Hugging Face models, retrieval-only) · **Keys:** none required. *Every extra package is import-guarded — if one is missing, that step is skipped with a note instead of crashing.*

# Week 11 · Notebook 4 — Modern RAG: Hybrid Retrieval + Reranking
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Plain vector search is the **starting point**. Here we add two of the 2026 upgrades from the lecture, kept deliberately small:
1. **Hybrid search** — combine **dense** (sentence-transformers) similarity with **BM25** keyword scores and **fuse** them, so we catch both paraphrases *and* exact terms.
2. **Reranking** — let a **cross-encoder** read each *(question, chunk)* pair together and reorder the shortlist.

> No OpenAI, no API key: embeddings + reranker are local Hugging Face models. Imports are guarded so missing packages degrade gracefully.

## 0. Install (all local, no keys)

In [ ]:
!pip -q install sentence-transformers rank-bm25

## 1. A tiny corpus
Same course handbook, pre-split into short passages so we can see the ranking move. Notice the exact tokens (`CSCI250`, `48-hour`) — those are where keyword search earns its keep.

In [ ]:
CHUNKS = [
    'Modules open Monday and the weekly lab is due Sunday at 11:59 PM Pacific.',
    'You may miss deadlines twice using the two automatic 48-hour extensions.',
    'Students may use AI assistants such as Claude and Gemini, but must disclose it.',
    'Exams, including the midterm and final, are AI-restricted.',
    'Labs are 50 percent, the midterm 20 percent, the final project 30 percent.',
    'The final project is due December 19.',
    'Office hours: Tuesday 5-6 PM and Saturday 10-11 AM Pacific on Zoom.',
    'Email the instructor with the subject prefix CSCI250 for a 48-hour reply.',
]
QUERY = 'How long are the automatic extensions and how do I email the instructor?'
print(len(CHUNKS), 'chunks; query =', QUERY)

## 2. Dense retrieval (embeddings)
Embed the query and every chunk with a local sentence-transformers model and rank by cosine similarity. Great at paraphrases; can blur exact tokens.

In [ ]:
import numpy as np

dense_scores = None
try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    emb = embedder.encode([QUERY] + CHUNKS, normalize_embeddings=True)
    qv, cv = emb[0], emb[1:]
    dense_scores = cv @ qv   # cosine sim (vectors are normalized)
    order = np.argsort(-dense_scores)
    print('DENSE ranking:')
    for r in order:
        print(f'  {dense_scores[r]:.3f} | {CHUNKS[r]}')
except Exception as e:
    print('[sentence-transformers unavailable, skipping dense:', e, ']')

## 3. BM25 keyword retrieval
BM25 is classic bag-of-words keyword scoring. It nails exact terms (`48-hour`, `CSCI250`) that dense embeddings smear together.

In [ ]:
bm25_scores = None
try:
    from rank_bm25 import BM25Okapi
    tokenized = [c.lower().split() for c in CHUNKS]
    bm25 = BM25Okapi(tokenized)
    bm25_scores = np.array(bm25.get_scores(QUERY.lower().split()))
    order = np.argsort(-bm25_scores)
    print('BM25 ranking:')
    for r in order:
        print(f'  {bm25_scores[r]:.3f} | {CHUNKS[r]}')
except Exception as e:
    print('[rank-bm25 unavailable, skipping BM25:', e, ']')

## 4. Fuse them → hybrid
Two rankers, two score scales — so we **min-max normalize** each to 0–1 and average. (In production a common alternative is *Reciprocal Rank Fusion*, which fuses by rank position instead of score.) If only one ranker ran, we fall back to it.

In [ ]:
def _norm(x):
    x = np.asarray(x, dtype=float)
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)

parts = [s for s in (dense_scores, bm25_scores) if s is not None]
if not parts:
    raise SystemExit('Neither retriever ran — install the packages above.')
hybrid = np.mean([_norm(p) for p in parts], axis=0)
hyb_order = list(np.argsort(-hybrid))
print(f'HYBRID ranking (fused {len(parts)} retriever(s)):')
for r in hyb_order:
    print(f'  {hybrid[r]:.3f} | {CHUNKS[r]}')

TOP_N = 4
shortlist = hyb_order[:TOP_N]
print('\nShortlist for reranking:', shortlist)

## 5. Rerank the shortlist with a cross-encoder
Bi-encoders (steps 2–4) embed the query and chunk **separately**. A **cross-encoder** reads the *(query, chunk)* pair **together**, so it scores relevance much more accurately — but it runs once per candidate, which is why we only rerank the shortlist, not the whole corpus. Guarded so a missing package just keeps the hybrid order.

In [ ]:
try:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    pairs = [(QUERY, CHUNKS[i]) for i in shortlist]
    ce_scores = reranker.predict(pairs)
    reranked = [i for _, i in sorted(zip(ce_scores, shortlist),
                                     key=lambda t: -t[0])]
    print('RERANKED top results (cross-encoder):')
    for s, i in sorted(zip(ce_scores, shortlist), key=lambda t: -t[0]):
        print(f'  {s:+.2f} | {CHUNKS[i]}')
    final = reranked
except Exception as e:
    print('[cross-encoder unavailable, keeping hybrid order:', e, ']')
    final = shortlist

print('\nFinal context order to feed the LLM:', final)

## 6. Takeaways
- **Hybrid** (dense + BM25) catches both paraphrases and exact tokens — fuse the two ranked lists.
- **Reranking** with a cross-encoder reorders the shortlist far more accurately than first-pass similarity; only rerank the top-k (it's slower per item).
- These are **additive upgrades** to Notebook 1's pipeline — add them only when the **RAG triad** (Notebook 3) says plain vector search is the bottleneck.
- Still 2026 baseline you didn't run here: **Contextual Retrieval** (prepend a context blurb to each chunk before embedding) and **agentic RAG** (the model decides when/what to retrieve) — see the lecture notes.

> **Responsible AI:** even with hybrid + reranking, RAG can still hallucinate — always cite sources, and remember retrieval **amplifies bias in your corpus**.